# TP XGBoost — Prédiction des retards de vol

**Objectif :** prédire si un vol arrive avec **plus de 15 minutes de retard** (`nycflights13`).

Support présenté en classe : à chaque trou `____`, la question est posée à la salle avant de montrer et d'expliquer la réponse. Les cellules 🟦 sont des **questions d'interprétation** à discuter ensemble.

In [ ]:
library(dplyr)
library(nycflights13)
library(xgboost)
library(pROC)
library(gbm)

## 1. Prétraitement

On crée la variable cible `retard` (1 si plus de 15 min de retard, 0 sinon) et on garde 50 000 vols.

In [ ]:
data(flights)

set.seed(123)
flights_clean <- flights |>
  filter(!is.na(arr_delay), !is.na(dep_delay), !is.na(air_time)) |>
  mutate(retard = ifelse(arr_delay > 15, 1, 0)) |>
  select(retard, dep_delay, air_time, distance, month, day, hour, minute) |>
  sample_n(50000)

cat("Proportion de vols en retard :", round(mean(flights_clean$retard), 4) * 100, "%")

> 🟦 **Question 1.** Environ 24 % des vols sont en retard. Les deux classes (retard / pas retard) sont-elles équilibrées ?
>
> *Réponds ici :*

## 2. Séparation train / test

**Question à la salle :** quelle proportion des données garde-t-on pour l'entraînement (et combien reste-t-il pour le test) ?

In [ ]:
set.seed(123)
idx <- as.vector(caret::createDataPartition(factor(flights_clean$retard), p = ____, list = FALSE))

train_flights <- flights_clean[idx, ]
test_flights  <- flights_clean[-idx, ]

X_train <- train_flights |> select(-retard) |> as.matrix()
y_train <- train_flights$retard
X_test  <- test_flights  |> select(-retard) |> as.matrix()
y_test  <- test_flights$retard

cat("Retards — train :", round(mean(y_train), 4) * 100, "%  |  test :", round(mean(y_test), 4) * 100, "%")

> 🟦 **Question 2.** La proportion de retards (~24 %) est identique dans le train et le test. Pourquoi est-ce une bonne chose ?
>
> *Réponds ici :*

## 3. Choisir le nombre d'arbres par validation croisée

**Question à la salle :** en combien de groupes (*folds*) découpe-t-on les données pour la validation croisée ?

In [ ]:
X_mat <- xgb.DMatrix(X_train, label = y_train)

set.seed(123)
sel_xgb <- xgb.cv(
  data       = X_mat,
  params     = list(objective = "binary:logistic", eta = 0.1),
  nrounds    = 300,
  metrics    = "auc",
  nfold      = ____,
  early_stopping_rounds = 10,
  verbose    = 0
)

ite_opt <- sel_xgb$best_iteration
cat("Nombre d'arbres retenu :", ite_opt)

## 4. Modèle final et évaluation sur le test

**Question à la salle :** quel nombre d'arbres utilise-t-on pour le modèle final ? (indice : on vient de le calculer juste au-dessus.)

In [ ]:
boost_opt <- xgboost(
  data = X_train, label = y_train,
  objective = "binary:logistic", eval_metric = "auc",
  eta = 0.1, nrounds = ____, verbose = 0
)

prev_opt <- predict(boost_opt, newdata = X_test)

roc_obj <- roc(y_test, prev_opt, quiet = TRUE)
cat("AUC test :", round(auc(roc_obj), 3), "\n")
cat("Accuracy test :", round(mean((prev_opt > 0.5) == y_test), 3))

## À retenir

- `retard` binaire (seuil 15 min) → classification binaire, classes déséquilibrées (~24 %).
- Split **stratifié** : même proportion de retards dans train et test.
- `xgb.cv` + `early_stopping_rounds` choisit le nombre d'arbres (`best_iteration`), qu'on réutilise dans le modèle final.
- On évalue sur `X_test` (jamais vu) : AUC ≈ 0,94.